# Notebook 02 – EDA after Cohort Extraction


--------

Cohort Population Extracted from 01 notebook

**Main Objectives**

+ Verify that data missingness reflects real-world medical patterns
+ Verify assumptions based off feature distributions and sample sizes
+ Baseline metrics are established and able to be used


In [23]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import missingno as msno
import duckdb

from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.metrics import roc_auc_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi' : 300})

sys.path.insert(0, '../src')



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from icu_tft.data.connect import get_connection

SQL_PATH = '../src/icu_tft/data/sql/mimic_iv_24h_icu_mortality_cohort.sql'
COHORT_PQ = '../data/processed/cohort.parquet'

con = get_connection()
with open(SQL_PATH) as f:
    cohort_sql = f.read()
    
con.execute(f'CREATE OR REPLACE TABLE cohort AS (\n{cohort_sql}\n)')
print('cohort table written to duckdb')

cohort = con.execute('SELECT * FROM cohort').df()
cohort.to_parquet(COHORT_PQ, index=False)

print(f'Cohort saved to {COHORT_PQ} shape={cohort.shape}')
print(cohort[['stay_id', 'mortality_24h', 'mortality_inhospital']].head())
    

[connect.py] Registered 30 views against mimic.duckdb
[connect.py] WARNING — 1 file(s) not found (views skipped):
  /Users/longer/ICU_Mortality_Prediction/data/raw/hosp/antimicrobial.csv.gz
cohort table written to duckdb
Cohort saved to ../data/processed/cohort.parquet shape=(67223, 14)
    stay_id  mortality_24h  mortality_inhospital
0  37081114              0                     0
1  37067082              0                     0
2  31205490              0                     0
3  37510196              0                     1
4  39060235              0                     0


In [ ]:

# architecture for synthetic data generation matching preset schema
# np.random.seed(617)
# n_patients = 1000
# stay_ids = np.arange(10000, 10000 + n_patients)
# cohort = pd.DataFrame(
#     {
#         'stay_id' : stay_ids,
#         'age' : np.random.normal(65, 15, n_patients).clip(18,100),
#         'gender' : np.random.choice(['M', 'F'], n_patients),
#         'race' : np.random.choice(['White', 'Black', 'Hispanic', 'Asian', 'Other', 'Unknown'], n_patients),
#         'insurance' : np.random.choice(['Medicare', 'Medicaid', 'Private', 'Other'], n_patients),
#         'mortality_24h' : np.random.binomial(1, 0.15, n_patients)
#     }
# )

# ts_records = []

# for sid, outcome in zip(cohort['stay_id'], cohort['mortality_24h']):
#     for t in range(24):
#         hr = np.random.normal(80 + (t*0.5 if outcome else 0), 15)
#         map_ = np.random.normal(85 - (t*0.5 if outcome else 0), 10)
#         lactate = np.random.exponential(1.5 + (t*0.1 if outcome else 0))
#         gcs = np.random.normal(14 - (t*0.2 if outcome else 0), 2).clip(3, 15)        
#         if np.random.rand() < 0.2: hr - np.nan
#         if np.random.rand() < 0.6: lactate - np.nan
        
#         ts_records.append([sid, t, hr, map_, lactate, gcs])
# ts = pd.DataFrame(ts_records, columns=['stay_id', 'time_step', 'heart_rate', 'mbp', 'lactate', 'gcs_total'])


## Cohort Data Flow Representation

In [26]:
base_cohort = con.execute('SELECT stay_id, mortality_24h, mortality_inhospital FROM cohort').df()

static_features_df = pd.read_parquet('../data/processed/static_features.parquet')

cols_to_drop = [c for c in static_features_df.columns if c.endswith('_y')]
static_features_df = static_features_df.drop(columns=cols_to_drop)
static_features_df.columns = static_features_df.columns.str.replace('_x', '', regex=False)
final_static_df = pd.merge(
    static_features_df,
    base_cohort,
    on='stay_id',
    how='inner'
)

print(final_static_df[['stay_id', 'mortality_24h']].head())

    stay_id  mortality_24h
0  37081114              0
1  37067082              0
2  31205490              0
3  37510196              0
4  39060235              0


In [ ]:
def draw_cohort_flow():
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('off')
    
    counts = [
        ('Total ICU Stays (MIMIC-IV)', 67223),
        ('Adults (Age >= 18)', 60769),
        ('First ICU Stay Only', 55000),
        ('ICU LOS >= 24h', 67223),
    ]

In [28]:
def create_demographic_table(df, target='mortality_24h'):
    survived = df[df[target] == 0]
    died = df[df[target] == 1]
    
    rows = []
    
    stat, p_val = mannwhitneyu(survived['anchor_age'], died['anchor_age'], alternative='two-sided')
    rows.append({
        'Feature': 'Age (Median [IQR])',
        f'Survived (n={len(survived)})': (
            f"{survived['anchor_age'].median():.1f} "
            f"[{survived['anchor_age'].quantile(0.25):.1f}-"
            f"{survived['anchor_age'].quantile(0.75):.1f}]"
        ),
        f'Died (n={len(died)})': (
            f"{died['anchor_age'].median():.1f} "
            f"[{died['anchor_age'].quantile(0.25):.1f}-"
            f"{died['anchor_age'].quantile(0.75):.1f}]"
        ),
        'P-Value': f'{p_val:.3f}'
    })
    
    cat_vars = ['gender', 'race', 'insurance']
    for var in cat_vars:
        rows.append({
            'Feature': f'**{var.capitalize()}**',
            f'Survived (n={len(survived)})': '',
            f'Died (n={len(died)})': '',
            'P-Value': ''
        })
        contingency = pd.crosstab(df[var], df[target])

        try:
            _, p_val, _, _ = chi2_contingency(contingency)
            p_str = f'{p_val:.3f}'
        except Exception:
            p_str = 'N/A'

        for i, val in enumerate(sorted(df[var].dropna().unique())):
            # FIX 5: was (survived[var] == var) — var is the column name, not the value
            surv_pct = (survived[var] == val).mean() * 100
            died_pct = (died[var]     == val).mean() * 100
            rows.append({
                'Feature': f'  {val}',
                f'Survived (n={len(survived)})': f"{(survived[var] == val).sum()} ({surv_pct:.1f}%)",
                f'Died (n={len(died)})':         f"{(died[var]     == val).sum()} ({died_pct:.1f}%)",
                'P-Value': p_str if i == 0 else ''
            })

    # FIX 6: return was inside the for-loop — dedented to function level
    table_df = pd.DataFrame(rows)
    return table_df


demo_table = create_demographic_table(cohort)
display(demo_table)

import pathlib
pathlib.Path('reports/figures').mkdir(parents=True, exist_ok=True)
demo_table.to_csv('reports/figures/02_demographics_table.csv', index=False)


,Feature,Survived (n=66472),Died (n=751),P-Value
0,Age (Median [IQR]),65.0 [53.0-76.0],69.0 [56.0-80.0],0.000
1,**Gender**,,,
2,F,28888 (43.5%),346 (46.1%),0.162
3,M,37584 (56.5%),405 (53.9%),
4,**Race**,,,
5,AMERICAN INDIAN/ALASKA NATIVE,140 (0.2%),0 (0.0%),0.000
6,ASIAN,770 (1.2%),11 (1.5%),
7,ASIAN - ASIAN INDIAN,185 (0.3%),2 (0.3%),
8,ASIAN - CHINESE,718 (1.1%),11 (1.5%),
9,ASIAN - KOREAN,60 (0.1%),0 (0.0%),
